## 全てのアルバム・フォルダを取得する。

In [1]:
from pathlib import Path
import os

# 1. ベースパスの設定（~ をフルパスに展開）
folder_base_path = "~/SharedFolder/junko/Google フォト/"
folder_path = Path(folder_base_path).expanduser()

# 2. ディレクトリ内を走査し、フォルダ（ディレクトリ）のみをリストに追加
album_names = [f.name for f in folder_path.iterdir() if f.is_dir()]

# 結果の確認
print(album_names)
print(f"Total albums: {len(album_names)}")

['羽生結弦', '2017 年の写真', '2020 年の写真', 'トリノとミラノへの旅行', '2016 年の写真', 'ドボ＆コナ', '北海道への旅行', '2026 年の写真', '2025 年の写真', 'イタリアへの旅行', '2014-08-05', 'スペイン＆フランス\u3000Voyage', 'レシピ', '石垣島＆西表島', 'Test', '2017\u3000スペイン＆フレンチバスク＆France南西地方＆パリ', '三兄弟', 'フランスへの旅行', '2023 年の写真', '2016年北イタリア、チンクエテッレ＆トスカーナ', '2014 年の写真', '2015 年の写真', '2018 年の写真', '🇹🇭バンコクの旅', '2022 年の写真', '姫路城', '２０１５年南イタリア、プーリア＆アマルフィの旅', 'France南西部の旅2014', '無題(1)', 'コート', '無題', '2026年1月22日〜30日\u3000ベトナム\u3000ホーチミンの旅', '2019 年の写真', '我が家の子供たち', '我が家の癒やし❤', '2021 年の写真', 'アーカイブ', '京都市', '2024 年の写真', 'ゆづ', '広島＆厳島神社', '金沢\u3000avec 八木ちゃん']
Total albums: 42


## jsonから画像ファイル名を取得する関数(未使用)。

In [2]:
import json
def find_image_from_json(json_path):
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
        # JSONの中にある "title" が、実際の画像ファイル名であることが多い
        original_name = data.get('title')
        return original_name

In [3]:
import os
import re

def rename_to_supplemental(filename):
    # ファイル名の末尾が "(数字).拡張子" になっているものを探す正規表現
    # グループ1: ファイル名本体, グループ2: (数字), グループ3: 拡張子
    pattern = re.compile(r"^(.*)(\(\d+\))(\.[^.]+)$")
    match = pattern.match(filename)
    new_name = filename
    if match:
        base_name = match.group(1)      # DSC_0001
        suffix_num = match.group(2)     # (1)
        extension = match.group(3)      # .JPG
        
        # 新しい名前の組み立て: DSC_0001.JPG.supplemental-metadata(1)
        new_name = f"{base_name}{extension}.supplemental-metadata{suffix_num}"
    return new_name




In [21]:
def convert_filename_custom(filename):
    return filename.replace(".naver.lin", ".naver.li")

## jpegからjsonファイル対応の作成

In [23]:
def jpg_to_json_map(jpg_files, json_files):

    jpg_db = {}
    for num,jpg_file in enumerate(jpg_files):
        # if num >= 1000:  # 一部ファイルだけ処理
        #     break
        jpg_name = jpg_file.stem  # 拡張子を除いたファイル名

        # 拡張子が ".jpg" (小文字) の場合のみ stem を取得、それ以外は name を取得
        if jpg_file.suffix == ".jpg":  jpg_name = jpg_file.stem
        elif jpg_file.suffix == ".png":  jpg_name = jpg_file.stem
        else:  jpg_name = jpg_file.name

        if jpg_name in json_files:
            json_file = json_files[jpg_name]
            jpg_db[jpg_file] = json_file
            # print(f"Match found: {jpg_file.name} <-> {json_file.name}")
        elif jpg_name[0:-1] in json_files:
            json_file = json_files[jpg_name[0:-1]]
            jpg_db[jpg_file] = json_file
            # print(f"Match found: {jpg_file.name} <-> {json_file.name}")
        elif convert_filename_custom(jpg_name) in json_files:
            json_file = json_files[convert_filename_custom(jpg_name)]
            jpg_db[jpg_file] = json_file
        else:
            # print(f"No JSON match for: {jpg_file.name}")
            # print(jpg_name)
            jpg_file_name =rename_to_supplemental(jpg_file.name)
            json_file = ModuleNotFoundError
            for key in json_files.keys():
                if key.find(jpg_file_name)==0:
                    json_file = json_files[key]
                    jpg_db[jpg_file] = json_file
                    # print(f"  Match found by prefix: {jpg_file.name} <-> {json_file.name}")
        
            if json_file ==None:
                    print(f"No JSON match for: {jpg_file.name}")

    print("JPG-JSON match num=",num+1)
    return jpg_db


In [24]:

from datetime import datetime, timezone
from pathlib import Path
import json

def get_photo_taken_datetime(json_file_path):

    # Helper function to convert timestamp to datetime
    def convert_timestamp(timestamp):
        return datetime.fromtimestamp(int(timestamp), tz=timezone.utc)

    with json_file_path.open('r') as json_file:
        json_data = json.load(json_file)
    
    # Extract relevant metadata
    creation_time = json_data.get('creationTime', {}).get('timestamp')
    photo_taken_time = json_data.get('photoTakenTime', {}).get('timestamp')
    
    # Convert timestamp to datetime
    # Googleフォトのライブラリに作成（アップロード）された日
    creation_datetime = convert_timestamp(creation_time) if creation_time else None
    # 写真が実際に撮影された日時です
    photo_taken_datetime = convert_timestamp(photo_taken_time) if photo_taken_time else None

    # print(f"Creation Time: {creation_datetime}")
    # print(f"Photo Taken Time: {photo_taken_datetime}")
    return  photo_taken_datetime


## 各アルバム内の画像ファイル(.jpg) を取得する。

In [28]:
from datetime import timedelta

# '我が家の子供たち'
# '2014 年の写真'
# 'France南西部の旅2014'
# '2019 年の写真'
# 'フランスへの旅行'
# '2015 年の写真'

# for album_no, album_name in enumerate(["Test"]):
for album_no, album_name in enumerate(album_names):

    jpg_files = []
    json_files = {}

    album_path = folder_path / album_name
    print(f"Album-{album_no+1}: {album_name}")
    for item in album_path.iterdir():
        if item.is_file():
            name, ext = os.path.splitext(item.name)
            if ext.lower() == '.jpg' or ext.lower() == '.png':
                # print(f"  File: {item.name}")
                jpg_files.append(item)
            elif ext.lower() == '.json':
                item_name = item.name.removesuffix(".json")
                item_name = item_name.removesuffix(".supplemental-metadata")
                # print(f"  JSON: {item_name}, {item.name}")
                name, ext = os.path.splitext(item.name) 
                json_files[item_name] = item
                json_files[item_name.removesuffix(".jpg")] = item

    print(f"Total JPG files: {len(jpg_files)}")
    print(f"Total JSON files: {len(json_files)}")

    jpg_db = jpg_to_json_map( jpg_files , json_files)

    num_changed =0
    for jpg_file, json_file in jpg_db.items():

        json_path =  album_path / json_file
        # print(json_path)

        # timedelta(hours=9) を足したことで、Googleフォトのシステム時間（UTC）から、
        # 私たちが日本で生活している時間（JST）へと正確に変換されます。
        # photo_taken_datetime = get_photo_taken_datetime(json_path) + timedelta(hours=9)
        photo_taken_datetime = get_photo_taken_datetime(json_path) 
        # print(f"Photo Taken Time: {photo_taken_datetime}")

        # 2. datetimeオブジェクトを「エポック秒（Unix時間）」に変換
        # OSが時間を扱うための数値形式にする必要があります
        timestamp = photo_taken_datetime.timestamp()
        
        jpg_path =  album_path / jpg_file
        # print(jpg_path)
        
        # 3. ファイルの「アクセス日時」と「更新日時」を書き換え
        # (アクセス日時, 更新日時) の順で指定します
        os.utime(jpg_path, (timestamp, timestamp))
        # print( f"更新完了: {jpg_file.name} -> {photo_taken_datetime}")
        num_changed += 1
    print(f"Total updated files: {num_changed}")          



Album-1: 羽生結弦
Total JPG files: 1
Total JSON files: 2
JPG-JSON match num= 1
Total updated files: 1
Album-2: 2017 年の写真
Total JPG files: 3416
Total JSON files: 5647
JPG-JSON match num= 3416
Total updated files: 3399
Album-3: 2020 年の写真
Total JPG files: 664
Total JSON files: 712
JPG-JSON match num= 664
Total updated files: 664
Album-4: トリノとミラノへの旅行
Total JPG files: 269
Total JSON files: 270
JPG-JSON match num= 269
Total updated files: 269
Album-5: 2016 年の写真
Total JPG files: 2013
Total JSON files: 3244
JPG-JSON match num= 2013
Total updated files: 2006
Album-6: ドボ＆コナ
Total JPG files: 221
Total JSON files: 408
JPG-JSON match num= 221
Total updated files: 220
Album-7: 北海道への旅行
Total JPG files: 31
Total JSON files: 32
JPG-JSON match num= 31
Total updated files: 31
Album-8: 2026 年の写真
Total JPG files: 576
Total JSON files: 596
JPG-JSON match num= 576
Total updated files: 573
Album-9: 2025 年の写真
Total JPG files: 1110
Total JSON files: 1145
JPG-JSON match num= 1110
Total updated files: 1106
Album-10: 